# Exit-action semantics: reproduce the tracked cohort analysis

**Status:** Exploratory · **Reproduction level:** R2 for the tracked exact-common-entry cohort and regenerated comparison figure; not a full replay · **Tier:** Docs-CI executable (CPU)

This notebook reads the concept-named evidence packet, verifies the cohort CSV against its evidence manifest, recomputes cohort statistics, regenerates a figure, and checks against the recorded summary. The notebook does not retrain or replay either checkpoint.

In [ ]:
import hashlib
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").is_file():
    ROOT = ROOT.parent
EVIDENCE = ROOT / "evidence/research/exit_action_semantics"
data_path = EVIDENCE / "common_entry_cohort.csv"
data = pd.read_csv(data_path)
summary = json.loads((EVIDENCE / "common_entry_cohort_summary.json").read_text())
provenance = json.loads((EVIDENCE / "run_identity.json").read_text())
manifest = json.loads((EVIDENCE / "evidence_manifest.json").read_text())
cohort_record = next(asset for asset in manifest["assets"] if asset["path"] == data_path.name)
cohort_sha256 = hashlib.sha256(data_path.read_bytes()).hexdigest()
assert cohort_sha256 == cohort_record["sha256"]
print(
    "cohort rows:", len(data), "recorded exact common entries:", summary["exact_common_entry_count"]
)
print("cohort sha256:", cohort_sha256)
print("provenance keys:", sorted(provenance)[:12])
print(
    "tracked outcome columns:",
    [c for c in data.columns if c.startswith("net_pnl_") or c.startswith("realized_r_")],
)

In [ ]:
recomputed = {
    "n": int(len(data)),
    "v2_sum_usd": float(data["net_pnl_v2"].sum()),
    "v3_sum_usd": float(data["net_pnl_v3"].sum()),
    "delta_sum_usd": float(data["delta_v3_minus_v2_usd"].sum()),
    "delta_mean_usd": float(data["delta_v3_minus_v2_usd"].mean()),
}
recorded = summary["exact_common_entry_pnl"]
print(recomputed)
assert recomputed["n"] == summary["exact_common_entry_count"]
assert abs(recomputed["v2_sum_usd"] - recorded["v2"]["sum_usd"]) < 1e-6
assert abs(recomputed["v3_sum_usd"] - recorded["v3"]["sum_usd"]) < 1e-6
assert abs(recomputed["delta_mean_usd"] - recorded["delta_v3_minus_v2"]["mean_usd"]) < 1e-6

Figure: summed tracked common-entry net PnL by entry month, with separate bars for v2 and v3.

In [ ]:
monthly = data.groupby("entry_month", sort=True)[["net_pnl_v2", "net_pnl_v3"]].sum()
monthly.plot(kind="bar", figsize=(8, 4), color=["#627d98", "#d07b45"])
plt.title("Tracked exact-common-entry cohort: summed net PnL")
plt.ylabel("USD")
plt.xlabel("Entry month")
plt.legend(["v2", "v3"])
plt.tight_layout()
plt.show()

## Result boundary

The 53 exact common entries are a small, post-divergence-selected subset from closed-loop replays. They do not isolate decoder effects causally or establish economic superiority. Full per-arm raw ledgers, raw tick data, and checkpoints are not included in this tracked packet, so this notebook reproduces only the tracked cohort summary and this figure.

See the [decoder behavior comparison](https://github.com/nntk072/Quantitative-RL-Trading-System/blob/main/evidence/research/exit_action_semantics/decoder_behavior_comparison.md), [result registry](result_registry), and [research methods](../research/evaluation_methodology).